# Notebook 01: Quickstart & Baseline Evaluation (NLP-28)
**Đề tài:** Sửa lỗi ngữ pháp & chuẩn hóa phụ đề ASR tiếng Việt bằng Seq2Seq

Notebook này so sánh trực tiếp hiệu năng giữa **Baseline (Luật & Từ điển)** và **Mô hình Seq2Seq** trên cùng một tập dữ liệu.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd

from src.data_processing.noise_generator import ASRNoiseGenerator
from src.baseline.rule_baseline import RuleAndDictionaryBaseline
from src.models.inference import Seq2SeqCorrector
from src.evaluation.metrics import evaluate_predictions
from src.srt_utils.srt_processor import SRTSubtitleProcessor

## 1. Sinh dữ liệu nhiễu nhân tạo ASR
Mô phỏng câu phụ đề bị lỗi: mất dấu câu, viết thường, phát âm ngọng.

In [ ]:
generator = ASRNoiseGenerator()
clean_sample = "Tôi đi làm ở ngân hàng và dành tiền tiết kiệm."
noisy_sample, errors = generator.apply_noise(clean_sample)

print(f"Câu gốc (Clean) : {clean_sample}")
print(f"Câu nhiễu (Noisy): {noisy_sample}")
print(f"Các lỗi đã áp dụng: {errors}")

## 2. Chạy thử nghiệm: Baseline vs Mô hình Seq2Seq

In [ ]:
# Khởi tạo Baseline
baseline = RuleAndDictionaryBaseline()
pred_baseline = baseline.predict(noisy_sample)

# Khởi tạo Mô hình Seq2Seq (ViT5 / BARTpho)
corrector = Seq2SeqCorrector(model_name_or_path="bmd1905/vietnamese-correction-v2")
pred_seq2seq = corrector.predict(noisy_sample)

print("=" * 60)
print(f"[Đầu vào Noisy]  : {noisy_sample}")
print(f"[1. Baseline sửa]: {pred_baseline}")
print(f"[2. Seq2Seq sửa] : {pred_seq2seq}")
print(f"[Mục tiêu Chuẩn] : {clean_sample}")
print("=" * 60)

## 3. Bảng so sánh chỉ số đo lường (Evaluation Comparison)
So sánh định lượng trực tiếp giữa Baseline và Seq2Seq theo đúng Rubric của thầy.

In [ ]:
sources = [noisy_sample]
references = [clean_sample]

# Chấm điểm Baseline
metrics_baseline = evaluate_predictions(sources, [pred_baseline], references)

# Chấm điểm Seq2Seq
metrics_seq2seq = evaluate_predictions(sources, [pred_seq2seq], references)

# Hiển thị bảng đối chứng rõ ràng
comparison_df = pd.DataFrame([
    {"Mô hình": "1. Baseline (Từ điển & Luật)", **metrics_baseline},
    {"Mô hình": "2. Seq2Seq Transformer", **metrics_seq2seq},
])

print("=== BẢNG SO SÁNH HIỆU NĂNG ĐỐI CHỨNG ===")
display(comparison_df)

## 4. Thử nghiệm trên tệp phụ đề .SRT thực tế
So sánh đầu ra của file phụ đề khi dùng Baseline vs khi dùng Seq2Seq.

In [ ]:
sample_srt = '''1\n00:00:01,000 --> 00:00:03,500\ntoi di lam o ngan hang va gianh tien tiet kiem\n\n2\n00:00:04,000 --> 00:00:07,000\nhom nay thoi tiet rat dep chung ta di choi\n'''

print("--- KẾT QUẢ KHI DÙNG BASELINE (Chỉ sửa được từ có sẵn trong từ điển) ---")
proc_base = SRTSubtitleProcessor(corrector_model=baseline)
print(proc_base.process_srt_string(sample_srt))

print("--- KẾT QUẢ KHI DÙNG SEQ2SEQ (Phục hồi toàn bộ dấu câu, viết hoa và ngữ pháp) ---")
proc_seq2seq = SRTSubtitleProcessor(corrector_model=corrector)
print(proc_seq2seq.process_srt_string(sample_srt))